# COVID-19 Global Data Analysis

## From Raw Data to Insights

### Project objective

This project analyzes reported COVID-19 cases, deaths, recoveries, and active cases over time and across reporting entities.

The main questions are:

- How did reported confirmed cases develop over time?
- When did the largest daily increase in reported cases occur?
- Which reporting entities recorded the highest numbers of confirmed cases and deaths?
- How did reported recoveries and active cases compare?
- Is there an association between reported confirmed cases and deaths?

> **Important:** The dataset combines country-level and province/state-level reporting. Therefore, results should be interpreted as analysis of the dataset's reported observations rather than perfectly standardized epidemiological estimates.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use("default")

## 2. Load the dataset

In [ ]:
# The notebook is stored in notebooks/ and the dataset in data/.
df = pd.read_csv("../data/covid_19_data.csv")

df.head()

## 3. Initial data inspection

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

### Initial observations

The dataset contains reporting observations by date and geographic entity. The original file includes an empty `Unnamed: 8` column, which will be removed during cleaning.

Missing values are expected in `Province/State` because some observations are reported at country level.

## 4. Data cleaning

In [ ]:
# Remove the empty column
df = df.drop(columns=["Unnamed: 8"])

# Convert date columns to datetime
df["ObservationDate"] = pd.to_datetime(df["ObservationDate"])
df["Last Update"] = pd.to_datetime(df["Last Update"], format="mixed")

# Create a general reporting-location field
df["Location"] = df["Province/State"].fillna(df["Country/Region"])

# Standardize column names
df = df.rename(columns={
    "SNo": "sno",
    "ObservationDate": "observation_date",
    "Province/State": "province_state",
    "Country/Region": "country_region",
    "Last Update": "last_update",
    "Confirmed": "confirmed",
    "Deaths": "deaths",
    "Recovered": "recovered"
})

df.info()

## 5. Feature engineering

In [ ]:
# Active cases represent confirmed cases not yet classified as recovered or deceased.
df["active"] = (
    df["confirmed"]
    - df["deaths"]
    - df["recovered"]
)

# Reported death rate at the observation level.
df["death_rate"] = np.where(
    df["confirmed"] > 0,
    df["deaths"] / df["confirmed"] * 100,
    0
)

df.head()

## 6. Global time-series analysis

### 6.1 Daily reported totals

For each observation date, reported confirmed cases, deaths, and recoveries are summed across the reporting observations in the dataset.

Because the source combines different geographic reporting levels, these should be interpreted as **dataset-level reported totals**, not perfectly standardized global epidemiological totals.

In [ ]:
daily_global = (
    df.groupby("observation_date")
      .agg(
          confirmed=("confirmed", "sum"),
          deaths=("deaths", "sum"),
          recovered=("recovered", "sum")
      )
      .reset_index()
)

daily_global["new_cases"] = daily_global["confirmed"].diff()
daily_global["new_deaths"] = daily_global["deaths"].diff()
daily_global["new_recovered"] = daily_global["recovered"].diff()

daily_global.head()

### 6.2 Global confirmed cases over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    daily_global["observation_date"],
    daily_global["confirmed"],
    linewidth=2
)

ax.set_xlabel("Date")
ax.set_ylabel("Reported confirmed cases")
ax.set_title("Global Reported Confirmed COVID-19 Cases Over Time")

ax.xaxis.set_major_locator(mdates.WeekdayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

**Interpretation:** The cumulative number of reported confirmed cases increased substantially during the study period, with a particularly steep increase during February 2020.

### 6.3 Daily reported new cases

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    daily_global["observation_date"],
    daily_global["new_cases"],
    linewidth=2
)

ax.set_xlabel("Date")
ax.set_ylabel("New reported cases")
ax.set_title("Daily Reported New COVID-19 Cases")

ax.xaxis.set_major_locator(mdates.WeekdayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

The daily series is more volatile than the cumulative series. It shows several increases and decreases in reported cases.

A daily increase represents a change in the **reported cumulative total**. It should not automatically be interpreted as the exact number of people who became infected on that date.

### 6.4 Peak daily reported increase

In [ ]:
peak_day = daily_global.loc[daily_global["new_cases"].idxmax()]
peak_day

The largest increase in reported confirmed cases occurred on **13 February 2020**, when the reported cumulative total increased by **15,148 cases**.

This large increase should be interpreted cautiously because changes in reporting, classification, or data revisions can create substantial one-day changes.

## 7. Reporting entity analysis

### 7.1 Confirmed cases

In [ ]:
country_cases = (
    df.groupby("country_region")["confirmed"]
      .max()
      .reset_index()
      .sort_values("confirmed", ascending=False)
)

country_cases.head(10)

In [ ]:
top_confirmed = country_cases.head(10).sort_values("confirmed")

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    top_confirmed["country_region"],
    top_confirmed["confirmed"]
)

ax.set_xlabel("Reported confirmed cases")
ax.set_ylabel("Reporting entity")
ax.set_title("Top 10 Reporting Entities by Reported Confirmed Cases")

for bar in bars:
    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f" {int(bar.get_width()):,}",
        va="center"
    )

plt.tight_layout()
plt.show()

**Interpretation:** Mainland China reported substantially more confirmed cases than the other reporting entities in this dataset. Italy, South Korea, and Iran formed a much smaller second tier.

### 7.2 Deaths

In [ ]:
country_deaths = (
    df.groupby("country_region")["deaths"]
      .max()
      .reset_index()
      .sort_values("deaths", ascending=False)
)

country_deaths.head(10)

In [ ]:
top_deaths = country_deaths.head(10).sort_values("deaths")

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    top_deaths["country_region"],
    top_deaths["deaths"]
)

ax.set_xlabel("Reported deaths")
ax.set_ylabel("Reporting entity")
ax.set_title("Top 10 Reporting Entities by Reported Deaths")

for bar in bars:
    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f" {int(bar.get_width()):,}",
        va="center"
    )

plt.tight_layout()
plt.show()

**Interpretation:** Mainland China had the highest reported number of deaths, followed by Italy and Iran.

### 7.3 Recoveries

In [ ]:
country_recovered = (
    df.groupby("country_region")["recovered"]
      .max()
      .reset_index()
      .sort_values("recovered", ascending=False)
)

country_recovered.head(10)

Mainland China also had the highest absolute number of reported recoveries. Absolute recovery counts should not be interpreted as a direct measure of healthcare performance because entities had very different numbers of reported cases and were at different stages of their outbreaks.

### 7.4 Combined reporting-entity summary

In [ ]:
country_summary = (
    df.groupby("country_region")
      .agg(
          confirmed=("confirmed", "max"),
          deaths=("deaths", "max"),
          recovered=("recovered", "max")
      )
      .reset_index()
)

country_summary["death_rate"] = np.where(
    country_summary["confirmed"] > 0,
    country_summary["deaths"] / country_summary["confirmed"] * 100,
    0
)

country_summary["recovery_rate"] = np.where(
    country_summary["confirmed"] > 0,
    country_summary["recovered"] / country_summary["confirmed"] * 100,
    0
)

country_summary["active"] = (
    country_summary["confirmed"]
    - country_summary["deaths"]
    - country_summary["recovered"]
)

country_summary.sort_values("confirmed", ascending=False).head(10)

### 7.5 Active cases

In [ ]:
country_summary.sort_values(
    "active",
    ascending=False
).head(10)

Mainland China had the largest number of reported active cases in absolute terms, followed by South Korea, Italy, and Iran.

Active-case comparisons should also be interpreted in the context of outbreak timing and reporting practices.

## 8. Death rate and recovery rate

The project calculates two descriptive rates:

- **Reported death rate:** deaths / confirmed × 100
- **Reported recovery rate:** recovered / confirmed × 100

These are descriptive ratios, not estimates of the true infection fatality rate or final probability of recovery. Small denominators can produce extreme percentages.

In [ ]:
country_summary.sort_values(
    "death_rate",
    ascending=False
).head(10)

In [ ]:
country_summary.sort_values(
    "recovery_rate",
    ascending=False
).head(10)

For example, an entity with one confirmed case and one recovery can show a 100% reported recovery rate. Therefore, recovery and death rates should be interpreted alongside the number of confirmed cases and the timing of the observations.

## 9. Confirmed cases vs. deaths

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    country_summary["confirmed"],
    country_summary["deaths"]
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Reported confirmed cases (log scale)")
plt.ylabel("Reported deaths (log scale)")
plt.title("Reported Confirmed Cases vs. Deaths by Reporting Entity")

plt.tight_layout()
plt.show()

In [ ]:
pearson_correlation = country_summary["confirmed"].corr(
    country_summary["deaths"]
)

print(f"Pearson correlation: {pearson_correlation:.3f}")

The Pearson correlation is **0.995**, indicating an extremely strong positive linear association between reported confirmed cases and reported deaths in this dataset.

However, **correlation does not establish causation**. Both variables are cumulative counts and are influenced by outbreak size, population, testing, reporting practices, timing, and other factors.

## 10. Key findings

### Finding 1 — Reported cases increased rapidly over time

The cumulative number of reported confirmed cases increased substantially during the study period. The daily reported series shows substantial volatility.

### Finding 2 — A major reporting spike occurred on 13 February 2020

The largest daily increase in the dataset was **15,148 reported cases** on 13 February 2020. This should be interpreted as a change in reported data rather than automatically as 15,148 newly infected people.

### Finding 3 — Mainland China dominated absolute case and death counts

Mainland China recorded the highest reported confirmed cases and deaths in the dataset, with Italy, South Korea, and Iran forming a second group by confirmed cases.

### Finding 4 — Active cases varied substantially across reporting entities

South Korea, Italy, and Iran had substantial numbers of reported active cases despite having far fewer confirmed cases than Mainland China.

### Finding 5 — Confirmed cases and deaths were strongly associated

The Pearson correlation between reported confirmed cases and deaths was **0.995**. This indicates a very strong association but does not demonstrate a causal relationship.

### Finding 6 — Rates require context

Reported death and recovery rates can be misleading when the number of confirmed cases is small or when entities are at different stages of their outbreaks.

## 11. Limitations

This analysis has several important limitations:

1. **Mixed geographic reporting levels:** The dataset contains country-level observations as well as province/state-level observations.
2. **Reporting changes:** Sudden changes in reported totals may reflect revisions, classification changes, or reporting practices.
3. **Timing effects:** Confirmed cases, deaths, and recoveries occur at different points in the disease process.
4. **Testing differences:** Confirmed cases depend on testing and case-detection practices.
5. **Cumulative-rate limitations:** Death and recovery rates calculated from cumulative reported counts do not represent final clinical outcomes.
6. **Small denominators:** Very small numbers of confirmed cases can produce extreme rates.
7. **Correlation vs. causation:** The strong relationship between confirmed cases and deaths should not be interpreted as proof of causality.

## 12. Conclusion

This project demonstrates a complete exploratory data-analysis workflow using a historical COVID-19 dataset: loading and inspecting raw data, cleaning inconsistent fields, creating derived metrics, aggregating observations, analyzing time trends, comparing reporting entities, calculating descriptive rates, and evaluating the relationship between reported cases and deaths.

The analysis shows substantial growth and volatility in reported cases during the study period, strong concentration of reported cases and deaths in Mainland China, and a very strong association between reported confirmed cases and deaths.

The results should be interpreted as an analysis of reported data rather than as definitive epidemiological estimates.

## 13. Reproducibility

Expected repository structure:

```text
covid-data-analysis/
├── data/
│   └── covid_19_data.csv
├── notebooks/
│   └── covid_analysis.ipynb
├── outputs/
├── report/
├── README.md
└── requirements.txt
```

The notebook expects the dataset at `../data/covid_19_data.csv` when executed from the `notebooks/` directory.